# Aula 5 – Arquiteturas Multicore, Cache e NUMA (WSL)
## Computação Paralela – IA

Experimentos:
1. Threads vs Processos (GIL)
2. Localidade de memória (Cache)
3. False Sharing
4. Afinidade de CPU (WSL/Linux)


---
# Exercício 1 – Threads vs Processos

In [ ]:
import os, time, math
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

def cpu_heavy(n):
    s = 0.0
    for i in range(1, n):
        s += math.sqrt(i) * math.sin(i)
    return s

def bench(executor_cls, workers, tasks=8, n=200000):
    t0 = time.perf_counter()
    with executor_cls(max_workers=workers) as ex:
        list(ex.map(cpu_heavy, [n]*tasks))
    return time.perf_counter() - t0

print('CPU count:', os.cpu_count())
for w in [1,2,4,8]:
    print(f'workers={w}', 'threads=', round(bench(ThreadPoolExecutor,w),3),
          'processes=', round(bench(ProcessPoolExecutor,w),3))


---
# Exercício 2 – Localidade de Memória

In [ ]:
import numpy as np
import time

n = 10000000
a = np.random.rand(n)

t0 = time.perf_counter()
a.sum()
t_seq = time.perf_counter() - t0

idx = np.random.permutation(n)
t0 = time.perf_counter()
a[idx].sum()
t_rand = time.perf_counter() - t0

print('Sequencial:', round(t_seq,3))
print('Aleatório:', round(t_rand,3))
print('Razão:', round(t_rand/t_seq,2))


---
# Exercício 3 – False Sharing

In [ ]:
import time, ctypes
from multiprocessing import Process, Array

ITERS = 3000000

def worker(shared, idx):
    for _ in range(ITERS):
        shared[idx] += 1

def run(padding):
    shared = Array(ctypes.c_longlong, (1+padding)*2, lock=False)
    i0 = 0
    i1 = 1+padding

    p1 = Process(target=worker, args=(shared,i0))
    p2 = Process(target=worker, args=(shared,i1))

    t0 = time.perf_counter()
    p1.start(); p2.start()
    p1.join(); p2.join()
    return time.perf_counter() - t0

print('Sem padding:', round(run(0),3))
print('Com padding:', round(run(16),3))


---
# Exercício 4 – Afinidade de CPU (Linux/WSL)

## Rodar o M2A1_Usa_CPU.ipynb em paralelo com a célula abaixo.

In [ ]:
# Tamanho para ocupar ~256KB a 512KB (comum para Cache L2)
SIZE = 64 * 1024  
ITERATIONS = 1000000

def cache_stresser(affinity_core=None):
    if affinity_core is not None:
        os.sched_setaffinity(0, {affinity_core})
    
    x = 0.5
    t0 = time.perf_counter()
    for i in range(ITERATIONS * 5):
        # Operação que não pode ser facilmente otimizada ou "jogada" para cache
        # Envolve trigonometria e condicionais (decisões do núcleo)
        if x > 0:
            x = np.sin(x) + np.cos(i)
        else:
            x = np.tan(x) - np.exp(-i)
            
    dt = time.perf_counter() - t0
    print(f"Afinidade {affinity_core}: {dt:.4f}s")

if __name__ == "__main__":
    print("--- Teste de Stress de Cache e Afinidade ---")
    
    # Caso 1: Com afinidade (trava no Core 0)
    p1 = Process(target=cache_stresser, args=(1,))
    p1.start()
    p1.join()

    # Caso 2: Sem afinidade (SO decide)
    # Para forçar o SO a mover o processo, você pode abrir 
    # outros programas pesados enquanto este roda.
    p2 = Process(target=cache_stresser, args=(None,))
    p2.start()
    p2.join()